# Notebook 04b: Post-Hoc Classifier Fine-Tuning (`04b_cmf_posthoc.ipynb`)

### Objectives & Workflow:
1. **Start from Best 04a Checkpoint**: Load representation-unlearned model from Notebook 04a across models (`resnet18` / `vit11m`) and seeds `[0, 1, ..., 9]`.
2. **Freeze Feature Encoder Invariantly**: Freeze encoder parameters (`requires_grad = False`) and set `encoder.eval()` to lock BatchNorm/LayerNorm running stats.
3. **Invariance Verification**: Validate that the encoder weights remain bitwise identical and extracted features have zero delta.
4. **Trainable Classifier Head**: Train only the linear classifier head to suppress output-level forget accuracy while maintaining retain accuracy.
5. **Validation-Driven Selection & Test Evaluation**: Optimize output accuracy on validation set and evaluate final model once on test set.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Post-Hoc Classifier Training Loop (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

STAGES = [
    {"lr": 0.01, "max_epochs": 5, "patience": 2},
    {"lr": 0.001, "max_epochs": 3, "patience": 2},
]
DELTA_RETAIN = 0.05
MIN_DELTA = 0.002

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

split_file = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact missing: {split_file}. Run Notebook 01 first.")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Retain dataset for head fine-tuning
retain_dataset = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_r_loader = DataLoader(retain_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

for seed in MODEL_SEEDS:
    ckpt_04a = f"./artifacts/checkpoints/cmf_04a_{MODEL_NAME}_seed_{seed}.pt"
    orig_metrics_file = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    
    if not os.path.exists(ckpt_04a):
        raise FileNotFoundError(f"04a checkpoint missing: {ckpt_04a}. Run Notebook 04a first.")
        
    with open(orig_metrics_file, "r") as f:
        orig_val_retain = json.load(f)["validation_metrics"]["output_retain"]
        
    print(f"\n==================================================")
    print(f"--- Running 04b: Post-Hoc Classifier | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"==================================================")
    
    # Load 04a model
    model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt_04a, map_location=device))
    
    # --- Freeze Encoder & Verify Invariance ---
    for p in model.encoder.parameters():
        p.requires_grad = False
    model.encoder.eval()
    
    # Unfreeze classifier head
    for p in model.head.parameters():
        p.requires_grad = True
        
    # Cache initial encoder state dict for invariance verification
    init_encoder_state = copy.deepcopy(model.encoder.state_dict())
    
    criterion = nn.CrossEntropyLoss()
    best_snapshot = None
    start_time = time.time()
    
    # --- Multi-Stage Head Fine-Tuning Loop ---
    for stage_idx, stage in enumerate(STAGES):
        if best_snapshot is not None:
            model.head.load_state_dict(best_snapshot["head_state"])
            
        optimizer = optim.SGD(model.head.parameters(), lr=stage["lr"], momentum=0.9, weight_decay=5e-4)
        no_gain = 0
        
        for epoch in range(1, stage["max_epochs"] + 1):
            model.encoder.eval()
            model.head.train()
            
            for images, targets in train_r_loader:
                images, targets = images.to(device), targets.to(device)
                optimizer.zero_grad()
                with torch.no_grad():
                    feats = model.encoder(images)
                out = model.head(feats)
                loss = criterion(out, targets)
                loss.backward()
                optimizer.step()
                
            # Validation evaluation
            val_metrics = evaluate_three_tier_metrics(
                model=model, train_loader=train_r_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
            )
            
            # 04b Score: Suppress output forget accuracy while maximizing retain accuracy
            score = (1.0 - val_metrics["output_forget"]) + val_metrics["output_retain"]
            eligible = (val_metrics["output_retain"] >= orig_val_retain - DELTA_RETAIN)
            
            print(f"Stage {stage_idx+1} Ep {epoch} | Val Retain: {val_metrics['output_retain']:.4f} | Output Forget: {val_metrics['output_forget']:.4f} | Score: {score:.4f} (Eligible: {eligible})")
            
            if eligible and (best_snapshot is None or score > best_snapshot["score"] + MIN_DELTA):
                best_snapshot = {"score": score, "metrics": val_metrics, "head_state": copy.deepcopy(model.head.state_dict())}
                no_gain = 0
            else:
                no_gain += 1
                
            if no_gain >= stage["patience"]:
                print(f"Early stopping triggered in stage {stage_idx+1}")
                break
                
    # Restore best head state
    if best_snapshot is not None:
        model.head.load_state_dict(best_snapshot["head_state"])
        
    # --- Invariance Verification Check ---
    curr_encoder_state = model.encoder.state_dict()
    for k in init_encoder_state:
        diff = torch.norm(init_encoder_state[k].float() - curr_encoder_state[k].float()).item()
        assert diff == 0.0, f"Encoder invariance violated on layer {k} with diff {diff}"
    print("\n[Invariance Check PASSED]: Encoder weights strictly identical to 04a.")
    
    runtime = time.time() - start_time
    save_ckpt = f"./artifacts/checkpoints/cmf_04b_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(model.state_dict(), save_ckpt)
    
    # Final evaluation on test set
    test_metrics = evaluate_three_tier_metrics(
        model=model, train_loader=train_r_loader, retain_eval_loader=test_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    
    res = {
        "variant": "04b_cmf_posthoc",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "runtime_sec": runtime,
        "validation_metrics": best_snapshot["metrics"] if best_snapshot else val_metrics,
        "test_metrics": test_metrics
    }
    res_path = f"./artifacts/metrics/cmf_04b_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(res, f, indent=2)
        
    print(f"04b Test Retain Output Acc: {test_metrics['output_retain']:.4f} | Forget: {test_metrics['output_forget']:.4f}")
    print(f"04b Test LP Forget Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
    print(f"04b Test NCC Forget Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
    print(f"Saved 04b checkpoint & metrics to {res_path}")